# 24 — Target networks: a copy that lags

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/024-dqn.ipynb)

Q-learning moves each value toward a target, and that target is built from the very values being moved. A target network breaks the loop by bootstrapping from an old copy of the weights. You will train the same toy with and without one, drag the refresh interval, and then pick a step size where the difference stops being cosmetic.

## 1. The numbers: one state, one target

The toy is a five-state corridor: every step moves right and pays a small reward, and state 4 is terminal. The model is a straight line `w0 + w1*s/3`, so two weights have to describe all five values.

In [ ]:
import numpy as np                                   # the array library
gamma = 0.99                                         # future reward counts almost as much as now
rewards = np.array([1.03, 1.02, 1.01, 1.00, 0.0])    # paid on leaving each state; state 4 is terminal
values = np.array([4., 3., 2., 1., 0.])              # the true value of each state
w = np.zeros(2)                                      # the model: w[0] + w[1] * s / 3
feat = lambda s: np.array([1., s / 3.])              # the two features of state s
print("value of state 2 now:", feat(2) @ w)          # 0.0 before any update
target = rewards[2] + gamma * (feat(3) @ w)          # the target: reward plus the discounted next value
print("target for state 2:", round(target, 3))       # 1.01 + 0.99 * 0.0
print("TD error:", round(target - feat(2) @ w, 3))   # what the update will shrink

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads the array library.</li><li><code>gamma = 0.99</code> — the discount on future reward.</li><li><code>rewards = ...</code> — the reward paid on leaving each state.</li><li><code>values = ...</code> — the true value, known here only so we can measure the error.</li><li><code>w = np.zeros(2)</code> — the whole model, two numbers.</li><li><code>feat = lambda s: ...</code> — a function turning a state into its two features.</li><li><code>print("value of state 2 now:", ...)</code> — the prediction before learning.</li><li><code>target = ...</code> — reward plus the discounted value of the next state.</li><li><code>print("target ...")</code> — one step of lookahead knows nothing about the far end yet.</li><li><code>print("TD error ...")</code> — the gap the update closes.</li></ul>

</details>

## 2. Step by step: one update at a time

Each step picks a state, builds a target from the frozen copy, and nudges the weights toward it. The copy is refreshed every `sync` steps; a large `sync` means a long-frozen target. Here `sync = 10`, and the printed error is measured across all five states.

In [ ]:
def train(lr, sync, steps=500):                       # one run; returns the error after each step
    rng = np.random.default_rng(0)                   # fixed seed, so runs are comparable
    w = np.zeros(2)                                  # the online weights
    wbar = w.copy()                                  # the frozen copy used for targets
    errs = []                                        # one error per update
    for t in range(steps):                           # the loop over updates
        s = int(rng.integers(0, 4))                  # a state the agent visits
        nxt = s + 1                                  # its only action: move right
        boot = 0. if nxt >= 4 else feat(nxt) @ wbar  # state 4 is terminal, so it contributes zero
        y = rewards[s] + gamma * boot                # the Bellman target
        w = w - lr * ((feat(s) @ w) - y) * feat(s)   # one gradient step on the TD error
        if t % sync == 0:                            # every sync steps
            wbar = w.copy()                          # refresh the frozen copy
        pred = np.array([feat(k) @ w for k in range(5)])   # the model's value for all five states
        errs.append(float(np.abs(pred - values).max()))    # worst error in the table
    return np.array(errs)                            # the curve
e10 = train(0.5, 10)                                 # lr 0.5, target refreshed every 10 steps
for t in [0, 50, 150, 299, 499]:                     # five checkpoints
    print(f"step {t:3d}  worst error {e10[t]:.5f}")  # watch it fall

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>def train(lr, sync, steps=500):</code> — the whole experiment in one function.</li><li><code>rng = np.random.default_rng(0)</code> — a seeded random source.</li><li><code>w = np.zeros(2)</code> — the online weights start at zero.</li><li><code>wbar = w.copy()</code> — a copy for building targets.</li><li><code>errs = []</code> — the error curve.</li><li><code>for t in range(steps):</code> — one update per iteration.</li><li><code>s = int(rng.integers(0, 4))</code> — pick a non-terminal state.</li><li><code>nxt = s + 1</code> — the successor.</li><li><code>boot = ...</code> — the frozen copy's value of the successor, or zero past the end.</li><li><code>y = ...</code> — reward plus discounted bootstrap.</li><li><code>w = w - lr * ...</code> — move the weights toward the target.</li><li><code>if t % sync == 0:</code> — the freezing schedule.</li><li><code>wbar = w.copy()</code> — refresh the copy.</li><li><code>pred = ...</code> — value of every state under the current weights.</li><li><code>errs.append(...)</code> — the worst of the five errors.</li><li><code>return np.array(errs)</code> — hand the curve back.</li><li><code>e10 = train(0.5, 10)</code> — run it once with a ten-step refresh.</li><li><code>for t in [...]:</code> — a few checkpoints.</li><li><code>print(...)</code> — error at each checkpoint.</li></ul>

</details>

## 3. Plot: with a target network and without

`sync = 1` refreshes the copy after every update, which is the same as having no target network at all. `sync = 10` keeps the target still for ten steps, so the weights move in short staircases.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
e1 = train(0.5, 1)                                   # no freeze: the target is rebuilt every step
plt.figure(figsize=(6, 3))                           # one wide panel
plt.plot(e1, label="sync = 1 (no target network)")   # the curve that chases itself
plt.plot(e10, label="sync = 10 (frozen copy)")       # the curve from section 2
plt.yscale("log")                                    # log y keeps both ends visible
plt.xlabel("update number")                          # x-axis label
plt.ylabel("worst value error (log)")                # y-axis label
plt.legend()                                         # name the two curves
plt.title("the frozen copy climbs in stairs and arrives a little later")   # takeaway
plt.show()                                           # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — loads the plotting library.</li><li><code>e1 = train(0.5, 1)</code> — the same run with the target refreshed every step.</li><li><code>plt.figure(...)</code> — one panel.</li><li><code>plt.plot(e1, ...)</code> — the unfrozen curve.</li><li><code>plt.plot(e10, ...)</code> — the frozen curve from section 2.</li><li><code>plt.yscale("log")</code> — errors span several orders of magnitude.</li><li><code>plt.xlabel("update number")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend()</code> — say which curve is which.</li><li><code>plt.title(...)</code> — the sentence the figure is making.</li><li><code>plt.show()</code> — render.</li></ul>

</details>

## 4. Interactive: how long to freeze

The slider is the refresh interval. Small values keep the target honest but jumpy; large values give clean staircases that start late. The title reports the error at the last update.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function
def show(sync=10):                                   # sync comes from the slider
    e = train(0.5, sync)                             # one full run at this interval
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(e, color="tab:orange")                  # the error curve
    plt.yscale("log")                                # log y
    plt.xlabel("update number")                      # x-axis label
    plt.ylabel("worst value error (log)")            # y-axis label
    plt.title(f"sync = {sync}: final error {e[-1]:.1e}")   # live readout
    plt.show()                                       # draw it
    print(f"sync {sync:2d} -> final error {e[-1]:.2e}")    # the same number as text
interact(show, sync=(1, 80, 1))                      # the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the slider widget.</li><li><code>def show(sync=10):</code> — one function per slider position.</li><li><code>e = train(0.5, sync)</code> — rerun training at this interval.</li><li><code>plt.figure(...)</code> — a new figure per drag.</li><li><code>plt.plot(e, ...)</code> — the error curve.</li><li><code>plt.yscale("log")</code> — log y.</li><li><code>plt.xlabel("update number")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.title(...)</code> — the final error, printed on the plot.</li><li><code>plt.show()</code> — draw it.</li><li><code>print(...)</code> — the same number as text.</li><li><code>interact(show, sync=(1, 80, 1))</code> — the slider, its range and its step.</li></ul>

</details>

## 5. When it breaks: the target that follows the weights

Keep the data and the update, and raise the step size to 0.9. With a frozen copy the run still lands on the answer. With a target rebuilt from the same weights every step, each update moves the goalposts, the next target is bigger, and the error climbs instead of falling.

In [ ]:
bad = train(0.9, 1)                                  # big step, target rebuilt every step
good = train(0.9, 10)                                # big step, but the target is frozen
plt.figure(figsize=(6, 3))                           # one panel
plt.plot(bad, label="sync = 1 (no freeze)")          # the run that diverges
plt.plot(good, label="sync = 10 (frozen copy)")      # the run that still converges
plt.yscale("log")                                    # log y: 1e12 and 1e-6 on one axis
plt.xlabel("update number")                          # x-axis label
plt.ylabel("worst value error (log)")                # y-axis label
plt.legend()                                         # name the two curves
plt.title("an unfrozen target feeds its own error back")   # takeaway
plt.show()                                           # render
print(f"no freeze ends at {bad[-1]:.1e}, frozen at {good[-1]:.1e}")   # orders of magnitude apart

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>bad = train(0.9, 1)</code> — step size 0.9, no freezing.</li><li><code>good = train(0.9, 10)</code> — same step size, ten-step freeze.</li><li><code>plt.figure(...)</code> — one panel.</li><li><code>plt.plot(bad, ...)</code> — the diverging curve.</li><li><code>plt.plot(good, ...)</code> — the converging curve.</li><li><code>plt.yscale("log")</code> — the only way to see both runs at once.</li><li><code>plt.xlabel("update number")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend()</code> — name the curves.</li><li><code>plt.title(...)</code> — the sentence the figure is making.</li><li><code>plt.show()</code> — render.</li><li><code>print(...)</code> — the two final errors, side by side.</li></ul>

</details>

## Your turn

Predict the largest `sync` in section 4 that still ends below an error of 0.01, then check it with the slider. In section 5, find a step size where the frozen run is also unstable, and explain what the freeze can and cannot protect you from.